## Retrieval-Augmented Generation (RAG)

Large Language Models (LLMs) generate responses primarily from knowledge acquired during training. Consequently, they do not automatically have access to specialised or private collections, such as a locally stored research corpus. **Retrieval-Augmented Generation (RAG)** addresses this limitation by combining an LLM with an external information retrieval system.[^1]

Instead of relying exclusively on the model's internal knowledge, a RAG system first **retrieves information relevant to the user's query** and then provides this information to the LLM as additional context. This is particularly useful when working with specialised corpora that were not part of the model's training data, or collections that are too large to fit into the model's context window.[^1]

### A simplified RAG pipeline can be represented as:

> **User question → retrieve relevant documents → add documents to the context → LLM → generated answer**

RAG does not normally retrain the language model on the external collection. Instead, the external data are made available to the model **at inference time**.[^1]




In [1]:
%pip install --upgrade --force-reinstall \
    "pydantic>=2.12,<2.13" \
    langchain \
    langchain-openai \
    langchain-chroma \
    langchain-docling \
    langchain-community \
    langchain-text-splitters \
    chromadb \
    docling \
    beautifulsoup4 \
    "numpy<2" \
    "pandas>=2.2,<3"

  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached langchain-1.4.0-py3-none-any.whl.metadata (6.2 kB)
  Using cached langchain_openai-1.6.2-py3-none-any.whl.metadata (3.4 kB)
  Using cached langchain_chroma-1.1.0-py3-none-any.whl.metadata (1.9 kB)
  Using cached langchain_docling-3.0.0-py3-none-any.whl.metadata (5.8 kB)
  Using cached langchain_community-0.4.2-py3-none-any.whl.metadata (3.4 kB)
  Using cached langchain_text_splitters-1.1.2-py3-none-any.whl.metadata (3.3 kB)
  Using cached chromadb-1.5.9-cp39-abi3-macosx_11_0_arm64.whl.metadata (5.0 kB)
  Using cached docling-2.127.0-py3-none-any.whl.metadata (11 kB)
  Using cached beautifulsoup4-4.15.0-py3-none-any.whl.metadata (3.8 kB)
  Using cached numpy-1.26.4-cp312-cp312-macosx_11_0_arm64.whl.metadata (61 kB)
  Using cached pandas-2.3.3-cp312-cp312-macosx_11_0_arm64.whl.metadata (91 kB)
  Using cached annotated_types-0.8.0-py3-none-any.whl.metadata (15 kB)
  Using cached pydantic_core-2.41.5-cp312-cp31

### Installations (skip if not neccessary)

In [3]:
import sys
!{sys.executable} -m pip install -U langchain-community

In [1]:
import importlib.metadata as md

for package in [
    "docling",
    "langchain-docling",
    "pydantic",
    "langchain",
    "numpy",
]:
    print(package, md.version(package))

from langchain_docling import DoclingLoader

print("Docling import successful")

docling 2.127.0
langchain-docling 3.0.0
pydantic 2.8.2
langchain 1.4.0
numpy 1.26.4


/opt/anaconda3/lib/python3.12/site-packages/pydantic/_internal/_fields.py:161: UserWarning: Field "model_impl" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/pydantic/_internal/_fields.py:161: UserWarning: Field "model_spec" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/pydantic/_internal/_fields.py:161: UserWarning: Field "model_name" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/pydantic/_internal/_fields.py:161: UserWarning: Field "model_version" has conflict with protected namespace "model_".

You may be able to

AttributeError: force_full_page_ocr

In [2]:
import sys
import importlib.metadata as md

print("Python:", sys.executable)

for package in ["docling", "langchain-docling", "pydantic", "pydantic-core"]:
    try:
        print(f"{package}: {md.version(package)}")
    except md.PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

Python: /opt/anaconda3/bin/python
docling: 2.127.0
langchain-docling: 3.0.0
pydantic: 2.12.5
pydantic-core: 2.41.5


In [4]:
import sys
import numpy as np

print(sys.executable)
print(np.__version__)
print(np.__file__)

/opt/anaconda3/bin/python
1.26.4
/opt/anaconda3/lib/python3.12/site-packages/numpy/__init__.py


In [3]:
import sys

!{sys.executable} -m pip install \
    --upgrade \
    --force-reinstall \
    --no-cache-dir \
    "pydantic==2.13.5"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 10.9 MB/s eta 0:00:00
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.16.0
    Uninstalling typing_extensions-4.16.0:
      Successfully uninstalled typing_extensions-4.16.0
  Attempting uninstall: annotated-types
    Found existing installation: annotated-types 0.6.0
    Uninstalling annotated-types-0.6.0:
      Successfully uninstalled annotated-types-0.6.0
  Attempting uninstall: typing-inspection
    Found existing installation: typing-inspection 0.4.4
    Uninstalling typing-inspection-0.4.4:
      Successfully uninstalled typing-inspection-0.4.4
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.20.1
    Uninstalling pydantic_core-2.20.1:
      Successfully uninstalled pydantic_core-2.20.1
  Attempting uninstall: pydantic
    Found existing installation: pydantic 2.8.2
    Uninstalling pydantic-2.8.2:
      Successfully uninstalled pydantic-2.8

In [1]:
import sys
import pydantic
import importlib.metadata as md

print("Python:", sys.executable)
print("Pydantic:", pydantic.__version__)
print("Pydantic location:", pydantic.__file__)
print("Docling:", md.version("docling"))
print("LangChain Docling:", md.version("langchain-docling"))

from langchain_docling import DoclingLoader

print("Docling import successful")

Python: /opt/anaconda3/bin/python
Pydantic: 2.13.5
Pydantic location: /opt/anaconda3/lib/python3.12/site-packages/pydantic/__init__.py
Docling: 2.127.0
LangChain Docling: 3.0.0
Docling import successful


In [2]:
import sys
!{sys.executable} -m pip check

thinc 8.3.6 has requirement numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4.
opencv-python 5.0.0.93 has requirement numpy>=2; python_version >= "3.9", but you have numpy 1.26.4.
streamlit 1.37.1 has requirement protobuf<6,>=3.20, but you have protobuf 7.35.1.


### Main imports

In [1]:
import os
import warnings
import logging

import bs4

from langchain.agents import AgentState, create_agent
from langchain.messages import MessageLikeRepresentation
from langchain.tools import tool

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_docling import DoclingLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

### Environment setup

For this step you will need to: 
- get a Langchain API Key (https://docs.langchain.com/oss/python/deepagents/rag)
- be added DHInfra project by Florian and get DHInfa API kez

In [2]:
from openai import OpenAI

In [3]:

os.environ["LANGCHAIN_API_KEY"] = "" # insert your own Langchain key
os.environ["LANGCHAIN_TRACING_V2"] = "false"  # <-- FIX 1: Disabled to prevent 403 error
os.environ["LANGCHAIN_PROJECT"] = "DHInfra-Tracing-Demo"
os.environ["LANGSMITH_DISABLE_RUN_COMPRESSION"] = "true"
os.environ["USER_AGENT"] = "my_agent"
os.environ["DHINFRA_API_KEY"] = "" # insert DHInfra key

In [68]:
# <-- FIX 2: Custom class to prevent the 422 "null content" error
class SanitizedChatOpenAI(ChatOpenAI):
    def _get_request_payload(self, input_, *args, **kwargs):
        payload = super()._get_request_payload(input_, *args, **kwargs)
        if "messages" in payload:
            for msg in payload["messages"]:
                if msg.get("content") is None:
                    msg["content"] = ""
        return payload

# Initialize chat model using the sanitized class
model = SanitizedChatOpenAI(
    model="qwen3.5-397b",
    openai_api_key=os.environ["DHINFRA_API_KEY"],
    openai_api_base="https://api.dhinfra.uni-graz.at/v1",
    model_kwargs={"parallel_tool_calls": False}
)

# Initialize embedding model
embeddings = OpenAIEmbeddings(
    model="qwen3-embedding-8b",
    openai_api_key=os.environ["DHINFRA_API_KEY"],
    openai_api_base="https://api.dhinfra.uni-graz.at/v1"
)

vector_store = Chroma(
    collection_name="migraanno_newspapers_v2",
    embedding_function=embeddings,
    persist_directory="./chroma_migraanno",
)


print("Chat model (Qwen), embedding model (Qwen3-Embedding-8B), and Chroma vector store setup done")

Chat model (Qwen), embedding model (Qwen3-Embedding-8B), and Chroma vector store setup done


## Adapting the code for our own data

If it is a dataframe:

In [5]:
import warnings
import logging
import os
import pandas as pd

warnings.filterwarnings("ignore")
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores.utils import filter_complex_metadata


In [6]:
import pandas as pd
from langchain_core.documents import Document

In [69]:
print("Documents in Chroma:", vector_store._collection.count())

Documents in Chroma: 96871


## Loading the embeddings (so no need for re-indexing)

In [70]:
from langchain_core.tools import tool


# Make the function available as a tool that the agent can call.
@tool
def retrieve_context(
    query: str,              # Semantic search query
    start_year: int = 1700,  # default value
    end_year: int = 1939,    # default value
    category: str = "",      # Optional exact category
    sentiment: str = "",     # Optional exact sentiment
    k: int = 4,              # Maximum number of retrieved passages
) -> str:
    """
    Retrieve historical newspaper passages with metadata.

    The query may be English, German, or multilingual.
    """

    # Create an inclusive year-range filter.
    conditions = [
        {"year": {"$gte": int(start_year)}},
        {"year": {"$lte": int(end_year)}},
    ]

    # Add an exact category filter if a category was supplied ($eq means “equal to.” It is a Chroma metadata-filter operator).
    if category:
        conditions.append({
            "category": {"$eq": category.strip()}
        })

    # Add an exact, lowercase sentiment filter if supplied.
    if sentiment:
        conditions.append({
            "sentiment": {"$eq": sentiment.strip().lower()}
        })

    # Filter the collection and rank matching documents
    # by semantic similarity to the query.
    retrieved_docs = vector_store.similarity_search(
        query=query,
        k=min(int(k), 10),  # Prevent retrieval of more than 10 passages
        filter={"$and": conditions},
    )

    # Return an informative message when nothing was found.
    if not retrieved_docs:
        return "No relevant newspaper passages were found."

    # Store the formatted documents that will be given to the agent.
    serialized_documents = []

    # Process each retrieved LangChain Document.
    for number, doc in enumerate(retrieved_docs, start=1):
        metadata = doc.metadata

        # Combine metadata and textual context into one readable source
        # serializes LangChain Document objects into a plain-text format for the language model
      
        # It is a custom human-readable text format constructed with a Python multiline f-string
        serialized_documents.append(
            f"""
SOURCE {number}

METADATA
ID: {metadata.get("chunk_id", "unknown")}
Topic: {metadata.get("topic", "unknown")}
Original label: {metadata.get("name_original", "unknown")}
Newspaper: {metadata.get("newspaper_title", "unknown")}
Date: {metadata.get("date", "unknown")}
Year: {metadata.get("year", "unknown")}
Category: {metadata.get("category", "unknown")}
Sentiment: {metadata.get("sentiment", "unknown")}
Relevancy probability: {metadata.get("relevancy_proba", "unknown")}

PRECEDING TEXT
{metadata.get("preceding_document", "")}

RETRIEVED TEXT
{doc.page_content}

FOLLOWING TEXT
{metadata.get("following_document", "")}
""".strip()
        )

    # Join all retrieved sources into one string for the agent.
    return "\n\n" + ("\n\n" + "=" * 80 + "\n\n").join(
        serialized_documents
    )

## Search newspapers function

In [71]:
query = (
    "What was the relationship between Croats and the government "
    "between 1860 and 1900?"
)

for step in agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What was the relationship between Croats and the government between 1860 and 1900?
================================== Ai Message ==================================
Tool Calls:
  retrieve_context (chatcmpl-tool-a76caf053ba5df37)
 Call ID: chatcmpl-tool-a76caf053ba5df37
  Args:
    query: Croats Croatian government Kroaten kroatisch Regierung Autonomie
    start_year: 1860
    end_year: 1900
    k: 4
================================= Tool Message =================================
Name: retrieve_context



SOURCE 1

METADATA
ID: 87510.0
Topic: 8
Original label: 8_ungarn_ungarischen_ungarische_ungarns
Newspaper: vtl
Date: 1878-07-01
Year: 1878
Category: MIN
Sentiment: neutral
Relevancy probability: 0.99881905

PRECEDING TEXT
* — Dem radicalen Oberst Denfert wirdein Denkmal errichtet, dessen Kosten durch Subscription in der Armee gedeckt werden sollen.

RETRIEVED TEXT
Topic: 8
Original label: 8_ungarn_ungarisc

In [76]:
query = (
    "Negative texts towards migration between 1900 and 1938 in the MIG category? "
    
)

for step in agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Negative texts towards migration between 1900 and 1938 in the MIG category? 
================================== Ai Message ==================================
Tool Calls:
  retrieve_context (chatcmpl-tool-99b7425b79c7535d)
 Call ID: chatcmpl-tool-99b7425b79c7535d
  Args:
    query: migration Einwanderung Auswanderung Zuwanderung Fremde Ausländer
    start_year: 1900
    end_year: 1938
    category: MIG
    sentiment: negative
    k: 4
================================= Tool Message =================================
Name: retrieve_context



SOURCE 1

METADATA
ID: 394073.0
Topic: 1710
Original label: 1710_ausländer_auswanderung_ausweisung_ausländern
Newspaper: aze
Date: 1915-04-16
Year: 1915
Category: MIG
Sentiment: negative
Relevancy probability: 0.999972

PRECEDING TEXT
Zunächst muß man sich ja selbstschützen! Der Richter sprach die angeklagten Frauen Blochs Fassel und Wilheim, serner die drei Kaffeehausan

In [77]:
query = (
    "How were Serbs discussed after 1914? "
    
)

for step in agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

How were Serbs discussed after 1914? 
================================== Ai Message ==================================
Tool Calls:
  retrieve_context (chatcmpl-tool-887a9cb69344bd8f)
 Call ID: chatcmpl-tool-887a9cb69344bd8f
  Args:
    query: Serbs Serben Serbia Serbien
    start_year: 1914
    end_year: 1920
    k: 4
================================= Tool Message =================================
Name: retrieve_context



SOURCE 1

METADATA
ID: 230127.0
Topic: 17
Original label: 17_serbien_serbischen_serbische_serben
Newspaper: dvb
Date: 1916-07-18
Year: 1916
Category: MIN
Sentiment: neutral
Relevancy probability: 0.99972194

PRECEDING TEXT
Auf den Feldern der Großgrundbesitzer arbeiten Frauen, Kinder und ältere Männer. Die Ernteaussichten in ganz Galizien sind sehr gute.

RETRIEVED TEXT
Topic: 17
Original label: 17_serbien_serbischen_serbische_serben
Category: MIN
Sentiment: neutral

[Austandsehung der 

In [79]:
query = (
    "How was migration discussed after 1900? "
    
)

for step in agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

How was migration discussed after 1900? 
================================== Ai Message ==================================
Tool Calls:
  retrieve_context (chatcmpl-tool-a4ba5b345e61c545)
 Call ID: chatcmpl-tool-a4ba5b345e61c545
  Args:
    query: migration Auswanderung Einwanderung Wanderung Übersiedlung emigration immigration
    start_year: 1900
    end_year: 1920
    k: 4
================================= Tool Message =================================
Name: retrieve_context



SOURCE 1

METADATA
ID: 352890.0
Topic: 20
Original label: 20_krieg_kriege_krieges_weltkrieg
Newspaper: aze
Date: 1912-09-17
Year: 1912
Category: CTX
Sentiment: negative
Relevancy probability: 0.9997818

PRECEDING TEXT
Auch wir englischen Sozialisten werden mit euch alles daran wenden, diese volksfeindliche Interessenpolitik einer Handvoll Kapitalisten auf das schärfste zu bekämpfen. Genau so wie in Deutschland hat auch in England 